In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configurações visuais
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("Bibliotecas carregadas!")

In [ ]:
df = pd.read_csv(r'D:/Análise_Empresas_Software/dataset_final.csv', dtype=str)

# Dicionário de situação cadastral
situacao_map = {
    '01': 'Nula',
    '02': 'Ativa',
    '03': 'Suspensa',
    '04': 'Inapta',
    '08': 'Baixada'
}

# Dicionário de CNAEs
cnae_map = {
    '6201500': 'Desenv. sob encomenda',
    '6201501': 'Desenv. sob encomenda',
    '6201502': 'Web design',
    '6202300': 'Software customizável',
    '6203100': 'Software não-customizável',
    '6204000': 'Consultoria em TI',
    '6209100': 'Suporte técnico/TI'
}

df['situacao_nome'] = df['situacao_cadastral'].map(situacao_map)
df['cnae_nome'] = df['cnae_fiscal_principal'].map(cnae_map)
df['ano_abertura'] = df['data_inicio_atividade'].str[:4]

print(f"Dataset carregado: {len(df):,} registros")
print(f"Colunas: {list(df.columns)}")

# Análise do Setor de Software Brasileiro
## Projeto de Extensão — Estruture Negócios | UFPB 2026

**Fonte dos dados:** Receita Federal do Brasil — Dados Abertos CNPJ (fev/2026)  
**Datasets utilizados:**
- Estabelecimentos (10 arquivos) — CNAE, UF, município, situação cadastral, data de abertura
- Simples Nacional — regime tributário das empresas

**CNAEs considerados:**
| Código | Descrição |
|--------|-----------|
| 6201500/01 | Desenvolvimento de software sob encomenda |
| 6201502 | Web design |
| 6202300 | Desenvolvimento de software customizável |
| 6203100 | Desenvolvimento de software não-customizável |
| 6204000 | Consultoria em tecnologia da informação |
| 6209100 | Suporte técnico e manutenção em TI |

In [ ]:
total = len(df)

print("=" * 55)
print("  SETOR DE SOFTWARE BRASILEIRO — fev/2026")
print("=" * 55)

print(f"\n{'Total de empresas':<35} {total:>10,}")
print(f"{'Estados cobertos':<35} {df['uf'].nunique():>10}")
print(f"{'Municípios cobertos':<35} {df['municipio'].nunique():>10,}")
print(f"{'Período de abertura':<35} {df['ano_abertura'].min()} – {df['ano_abertura'].max()}")

print("\n--- Situação Cadastral ---")
for k, v in df['situacao_nome'].value_counts().items():
    print(f"  {k:<20} {v:>8,}  ({v/total*100:.1f}%)")

print("\n--- Distribuição por CNAE ---")
for k, v in df['cnae_nome'].value_counts().items():
    print(f"  {k:<30} {v:>8,}  ({v/total*100:.1f}%)")

print("\n--- Simples Nacional ---")
for k, v in df['opcao_simples'].value_counts().items():
    label = 'Optante' if k == 'S' else 'Não optante'
    print(f"  {label:<20} {v:>8,}  ({v/total*100:.1f}%)")

print("\n--- Top 10 Estados ---")
for uf, v in df['uf'].value_counts().head(10).items():
    print(f"  {uf:<5} {v:>8,}  ({v/total*100:.1f}%)")

In [ ]:
df.head(10)

In [ ]:
colunas_relevantes = [
    'cnpj_basico',
    'nome_fantasia', 
    'uf',
    'municipio',
    'situacao_nome',
    'cnae_nome',
    'ano_abertura',
    'opcao_simples',
    'opcao_mei'
]

df[colunas_relevantes].head(10)

In [ ]:
# Remover registros EX e salvar dataset limpo
df = df[df['uf'] != 'EX']

print(f"Dataset após remoção de EX: {len(df)} empresas")
print(f"UFs únicas restantes: {sorted(df['uf'].unique())}")

# Análise: Distribuição de Empresas de Software Ativas por UF

## Visão Geral

O gráfico revela uma concentração geográfica extrema do setor de software brasileiro,
com o Sudeste e Sul dominando de forma absoluta enquanto o Nordeste permanece
fragmentado e sub-representado.

---

## 1. Dominância do Sudeste/Sul

São Paulo (SP) lidera com enorme distância — visualmente, sua barra ultrapassa
120.000 empresas ativas, superando sozinho a soma de praticamente todos os demais
estados. O grupo formado por MG, PR, RJ, SC e RS ocupa as posições seguintes,
todas com barras entre 15.000 e 25.000 empresas, configurando um bloco coeso de
alta concentração nas regiões Sudeste e Sul.

O Distrito Federal (DF) aparece como exceção fora desse bloco, posicionado em 7º
lugar com cerca de 8.000 empresas — reflexo da concentração de órgãos públicos
e empresas de tecnologia voltadas ao governo federal.

---

## 2. O Nordeste no Contexto Nacional

Os estados nordestinos (destacados em coral) aparecem dispersos ao longo da metade
inferior do ranking, sem nenhum representante no top 7. Os três estados nordestinos
com maior presença são:

| UF | Posição no ranking | Observação |
|----|--------------------|------------|
| CE | 8º | Polo tecnológico regional mais consolidado |
| BA | 10º | Maior estado nordestino em população e PIB |
| PE | 11º | Hub de tecnologia do Recife (Porto Digital) |

A Paraíba (PB) aparece na 14ª posição, abaixo de estados como GO (Centro-Oeste)
e ES (Sudeste), evidenciando a limitação do ecossistema local — contexto direto
da narrativa do Carlos e da hipótese central do projeto.

---

## 3. Cauda Longa e Estados Periféricos

A partir de MT, a distribuição entra em uma longa cauda com barras quase
imperceptíveis. Estados como AC, AP e RR têm presença mínima, o que levanta
questões metodológicas para análises per capita: a baixa densidade populacional
pode distorcer métricas de proporção nesses casos.

---

## 4. Insight Principal para o Projeto

A distribuição confirma visualmente a hipótese de desigualdade regional do
**Estruture Negócios**: o Nordeste, com ~28% da população brasileira, concentra
uma fração desproporcional e pequena das empresas de software ativas. Nenhum
estado nordestino entra sequer no top 7 nacional, e os três mais representativos
(CE, BA, PE) têm volumes próximos — sem um polo dominante que concentre o
ecossistema regional.

---

## 5. Próximos Passos Sugeridos

- **Concentração relativa**: calcular a participação percentual Nordeste vs. Sudeste
  no total de empresas ativas
- **Análise por porte**: verificar se o Nordeste tem proporcionalmente mais MEIs e
  menos EPPs/MEs que o Sudeste — o que indicaria menor maturidade empresarial
- **Evolução temporal**: cruzar com `ano_abertura` para ver se a desigualdade está
  se ampliando ou reduzindo ao longo do tempo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Distribuição por UF
uf_counts = df[df['situacao_nome'] == 'Ativa']['uf'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 12))

bars = ax.barh(uf_counts.index, uf_counts.values, color='steelblue')

# Destaca Nordeste
nordeste = ['AL','BA','CE','MA','PB','PE','PI','RN','SE']
for bar, uf in zip(bars, uf_counts.index):
    if uf in nordeste:
        bar.set_color('coral')

ax.set_title('Empresas de Software Ativas por UF', fontsize=14, fontweight='bold')
ax.set_xlabel('Quantidade de Empresas')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Legenda
from matplotlib.patches import Patch
legend = [Patch(color='steelblue', label='Outras regiões'),
          Patch(color='coral', label='Nordeste')]
ax.legend(handles=legend)

plt.tight_layout()
plt.show()

# Cell 7 — Concentração Regional de Empresas de Software Ativas

## Visão Geral

O gráfico de concentração regional expõe de forma direta a assimetria estrutural
do setor de software brasileiro: duas regiões concentram mais de 80% das empresas
ativas enquanto o restante do país divide os 20% restantes.

---

## Distribuição por Região

| Região | % do total | Observação |
|--------|-----------|------------|
| Sudeste | 66,8% | Dois terços do setor nacional |
| Sul | 17,0% | Segunda região, menos da metade da população nordestina |
| Nordeste | 8,4% | ~28% da população, menos de 1/10 das empresas |
| Centro-Oeste | 6,1% | Inflado pelo DF (governo + empresas de tecnologia pública) |
| Norte | 1,7% | Presença quase simbólica |

---

## O Número-Chave do Projeto

O Nordeste concentra aproximadamente **28% da população brasileira** mas apenas
**8,4% das empresas de software ativas**. Isso representa uma razão de cerca de
**1:3,3** — para cada empresa de software que existiria se a distribuição fosse
proporcional à população, existe menos de uma de fato.

Esse desequilíbrio é o argumento central do **Estruture Negócios** condensado
em um único número.

---

## Comparativo Sul vs. Nordeste

O Sul (17%) tem mais que o dobro da participação do Nordeste (8,4%), apesar de
ter população consideravelmente menor. Esse contraste é especialmente relevante
porque o Sul não possui vantagens históricas de capital político ou financeiro
comparáveis ao Sudeste — sua maior participação reflete diferenças em
infraestrutura tecnológica, cultura empreendedora e acesso a mercados.

---

## Implicações para a Narrativa

A concentração no Sudeste não é apenas um dado estatístico — ela se traduz em
menor acesso a clientes, investidores, eventos, parcerias e talentos para
empreendedores como o Carlos em João Pessoa. O mercado de software brasileiro
é geograficamente centralizado em um grau que não encontra paralelo em setores
como agronegócio ou varejo.

---

## Próximos Passos

- Cruzar com dados populacionais para calcular a densidade de empresas por
  100 mil habitantes por região
- Verificar se a concentração é ainda maior quando filtrada por empresas de
  maior porte (ME, EPP) — excluindo MEIs

In [ ]:
import matplotlib.pyplot as plt

regioes = {
    'Norte': ['AM','RR','AP','PA','TO','RO','AC'],
    'Nordeste': ['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
    'Centro-Oeste': ['MT','MS','GO','DF'],
    'Sudeste': ['SP','RJ','MG','ES'],
    'Sul': ['PR','SC','RS']
}

def get_regiao(uf):
    for regiao, ufs in regioes.items():
        if uf in ufs:
            return regiao
    return 'Outro'

df['regiao'] = df['uf'].apply(get_regiao)

dist = df['regiao'].value_counts()
pct  = (dist / dist.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 5))
cores = ['#4e79a7' if r != 'Nordeste' else '#f28e2b' for r in pct.index]
bars = ax.barh(pct.index, pct.values, color=cores)
ax.bar_label(bars, fmt='%.1f%%', padding=4)
ax.set_xlabel('% do total de empresas ativas')
ax.set_title('Concentração Regional de Empresas de Software Ativas')
ax.set_xlim(0, pct.max() + 10)
plt.tight_layout()
plt.show()

# Cell 8 — Taxa de Sobrevivência por Região

## Visão Geral

A taxa de empresas ativas por região revela um resultado contraintuitivo que
desafia narrativas simplistas sobre o ecossistema de software nordestino:
empresas do Nordeste sobrevivem mais que as do Sudeste.

---

## Dados por Região

| Região | Total de empresas | % Ativas |
|--------|------------------|----------|
| Sul | 81.570 | 58,5% |
| Norte | 7.971 | 58,5% |
| Nordeste | 40.313 | 57,4% |
| Centro-Oeste | 29.381 | 56,4% |
| Sudeste | 319.836 | 52,9% |

---

## O Resultado Contraintuitivo

O Nordeste (57,4%) supera o Sudeste (52,9%) em taxa de sobrevivência. A variação
total entre todas as regiões é de apenas ~6 pontos percentuais, o que indica que
**resiliência empresarial não é o problema do ecossistema nordestino**.

---

## Por Que o Sudeste Tem a Menor Taxa?

O volume massivo de empresas em SP e RJ inclui proporcionalmente mais:
- Empresas abertas de forma oportunista ou especulativa
- Negócios de curta duração ligados a contratos pontuais
- Tentativas de empreendedorismo em mercado mais saturado e competitivo

O maior acesso a capital e infraestrutura no Sudeste também reduz a barreira de
entrada — o que gera mais tentativas, e consequentemente mais encerramentos.

---

## O Insight Central para o Projeto

O problema do ecossistema nordestino **não é que as empresas fecham mais** —
é que **poucas são abertas**. A questão é estrutural: falta de acesso a mercado,
capital, talentos e infraestrutura reduz o número de tentativas, não a qualidade
das que existem.

Esse dado quebra um possível preconceito e fortalece a narrativa do Carlos:
empreendedores nordestinos de software são tão resilientes quanto os do Sudeste,
mas operam em um ambiente com muito menos recursos e oportunidades.

---

## Próximos Passos

- Detalhar os tipos de baixa por região (encerramento voluntário vs. inapta vs.
  suspensa) para entender melhor o perfil dos encerramentos
- Verificar se a taxa de sobrevivência varia por porte da empresa dentro do Nordeste

In [ ]:
# situacao_cadastral: 02 = Ativa, 03 = Suspensa, 04 = Inapta, 08 = Baixada
df_todas = df.copy()  # use o df SEM filtro de situação ativa se tiver, senão ok

sobrev = df_todas.groupby(['regiao', 'situacao_cadastral']).size().unstack(fill_value=0)
sobrev['total'] = sobrev.sum(axis=1)
sobrev['pct_ativa'] = (sobrev.get(2, sobrev.get('02', 0)) / sobrev['total'] * 100).round(1)

print(sobrev[['total', 'pct_ativa']].sort_values('pct_ativa', ascending=False))

sobrev['pct_ativa'].sort_values().plot(
    kind='barh', color='#f28e2b', figsize=(8, 4),
    title='Taxa de Empresas Ativas por Região (%)'
)
plt.xlabel('% ativas')
plt.tight_layout()
plt.show()

# Cell 9 — Perfil das Empresas de Software no Nordeste

## Visão Geral

O perfil das empresas nordestinas revela um ecossistema predominantemente voltado
a serviços de baixo valor agregado e sem escala, onde o modelo de negócio dominante
é a venda de horas de trabalho — não de produto.

---

## Simples Nacional

| Regime | % |
|--------|---|
| Fora do Simples Nacional | 57,3% |
| Simples Nacional | 42,7% |

A maioria das empresas nordestinas de software está fora do Simples Nacional.
Isso pode indicar dois cenários distintos: empresas que cresceram além do teto
de faturamento do regime (R$ 4,8M/ano) ou empresas que optaram pelo Lucro
Presumido por razões tributárias. Uma análise mais aprofundada com dados de
capital social ajudaria a distinguir os dois casos.

---

## Top 8 CNAEs — O Perfil Revela o Problema

| Posição | CNAE | Tipo de negócio |
|---------|------|-----------------|
| 1º | Suporte técnico/TI | Serviço — vende hora de trabalho |
| 2º | Desenvolvimento sob encomenda | Serviço — produto para um cliente |
| 3º | Consultoria em TI | Serviço — vende conhecimento/hora |
| 4º | Software customizável | Produto — algum potencial de escala |
| 5º | Software não-customizável | Produto — maior potencial de escala |
| 6º | Web design | Serviço — baixo valor agregado |

---

## O Ecossistema de Serviço vs. Produto

O Nordeste está preso em um modelo de **serviço**, não de **produto**:

**Suporte técnico/TI** e **Consultoria em TI** juntos representam os dois maiores
grupos em volume. Empresas nessas categorias têm crescimento atrelado à contratação
de mais pessoas — não escalam de forma exponencial. Para dobrar a receita, é
preciso dobrar o time.

**Desenvolvimento sob encomenda** (2º lugar) é um modelo híbrido: tecnicamente
produz software, mas para um cliente específico, sem propriedade intelectual
própria ou receita recorrente. Também não escala.

**Software customizável e não-customizável** — as categorias com real potencial
de produto — aparecem apenas em 4º e 5º lugar, com volumes significativamente
menores que os de serviço.

---

## Implicação para a Narrativa do Carlos

O Carlos do Estruture Negócios é estatisticamente mais provável de estar no
grupo de suporte/consultoria do que desenvolvendo um produto proprietário.
Isso não é falta de capacidade técnica — é reflexo de um mercado local que
demanda serviços imediatos e não tem capital de risco para financiar a construção
de produto. O empreendedor nordestino adapta sua oferta à demanda disponível.

---

## Próximos Passos

- Comparar o mix CNAE Nordeste vs. Sudeste: o Sudeste tem proporção maior de
  software não-customizável e SaaS?
- Cruzar CNAE com capital social médio para verificar se empresas de produto
  têm maior capitalização mesmo no Nordeste

In [ ]:
df_ne = df[df['regiao'] == 'Nordeste']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Porte (opcao_simples + porte implícito via capital)
simples = df_ne['opcao_simples'].value_counts()
simples.index = simples.index.map({'S': 'Simples Nacional', 'N': 'Fora do Simples'})
axes[0].pie(simples, labels=simples.index, autopct='%1.1f%%',
            colors=['#f28e2b','#4e79a7'])
axes[0].set_title('Opção pelo Simples Nacional — Nordeste')

# CNAE predominante
top_cnae = df_ne['cnae_nome'].value_counts().head(8)
top_cnae.sort_values().plot(kind='barh', ax=axes[1], color='#f28e2b')
axes[1].set_title('Top 8 CNAEs — Nordeste')
axes[1].set_xlabel('Nº de empresas')

plt.tight_layout()
plt.show()

# Cell 10 — Evolução Temporal de Abertura de Empresas por Região (2000–2025)

## Visão Geral

A série histórica de abertura de empresas de software revela que a digitalização
acelerada pós-2020 beneficiou todas as regiões — mas ampliou, e não reduziu,
a desigualdade regional absoluta.

---

## Três Períodos Distintos

### 2000–2019: Estagnação Generalizada

Todas as regiões mantiveram volumes relativamente estáveis por quase duas décadas.
O Nordeste oscilava em torno de 800 empresas abertas por ano, colado ao Norte e
Centro-Oeste. O Sudeste dominava com ~7.000–9.000 ao ano, mas também sem
crescimento expressivo. O setor de software brasileiro crescia de forma lenta
e concentrada.

### 2020: O Ponto de Inflexão

A pandemia funciona como catalisador simultâneo para todas as regiões. Três
fatores convergem:
- Digitalização forçada de negócios tradicionais aumenta a demanda por software
- Facilitação de abertura de MEI e ME reduz a barreira de entrada
- Trabalho remoto reduz (mas não elimina) a dependência de estar nos grandes centros

O resultado é um salto abrupto e sincronizado em todas as curvas a partir de 2020.

### 2020–2025: Crescimento com Gap Ampliado

O Nordeste (verde) sai de ~800 para ~5.000 empresas abertas por ano — um
crescimento real de aproximadamente 525%. O Sudeste vai de ~8.000 para ~30.000
no mesmo período — crescimento de ~275%, menor em termos relativos, mas muito
maior em termos absolutos.

---

## O Gap Está se Ampliando

| Período | Sudeste (aprox.) | Nordeste (aprox.) | Diferença absoluta |
|---------|-----------------|-------------------|-------------------|
| 2019 | ~8.000/ano | ~800/ano | ~7.200 |
| 2024 | ~21.000/ano | ~3.500/ano | ~17.500 |
| 2025 | ~30.000/ano | ~5.000/ano | ~25.000 |

A desigualdade regional **não está convergindo** na era digital — está se
aprofundando em termos absolutos. Mesmo com menos barreiras geográficas para
abertura de empresas de software, os recursos concentrados no Sudeste (capital,
clientes, talentos, acesso a investimento) continuam acelerando o crescimento
local de forma desproporcional.

---

## Atenção: Dado de 2025

O salto abrupto do Sudeste em 2025 merece verificação antes de ser usado na
narrativa. Empresas abertas muito recentemente podem ainda não ter situação
cadastral consolidada no dataset, o que poderia superestimar o volume de 2025.
Recomenda-se tratar 2025 como dado parcial nas conclusões do projeto.

---

## Implicação para o Projeto

A evolução temporal confirma que a desigualdade regional no setor de software
é **estrutural e persistente**, não um artefato histórico em processo de
correção natural. A digitalização acelerou o crescimento do setor em todo o
Brasil, mas o fez de forma desigual — reforçando a necessidade de políticas
específicas para o ecossistema nordestino que vão além da simples redução de
barreiras de entrada.

---

## Próximos Passos

- Calcular a taxa de crescimento anual composta (CAGR) por região entre 2020 e 2024
- Verificar se o crescimento nordestino pós-2020 está concentrado em CE/BA/PE
  ou se houve distribuição mais ampla entre os estados menores
- Filtrar 2025 ou tratá-lo separadamente nas análises conclusivas

In [ ]:
df['ano_abertura'] = pd.to_numeric(df['ano_abertura'], errors='coerce')
df_time = df[df['ano_abertura'].between(2000, 2025)]

evolucao = df_time.groupby(['ano_abertura', 'regiao']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
for regiao in ['Sudeste', 'Sul', 'Nordeste', 'Centro-Oeste', 'Norte']:
    if regiao in evolucao.columns:
        lw = 2.5 if regiao == 'Nordeste' else 1.2
        ls = '-' if regiao == 'Nordeste' else '--'
        ax.plot(evolucao.index, evolucao[regiao], label=regiao, linewidth=lw, linestyle=ls)

ax.set_title('Abertura de Empresas de Software por Região (2000–2025)')
ax.set_xlabel('Ano')
ax.set_ylabel('Nº de empresas abertas')
ax.legend()
plt.tight_layout()
plt.show()

# Bloco 1 — Concentração Municipal

## 1.1 — Top 10 municípios por região

## Visão Geral

Identifica os principais polos de software em cada região, com foco no Nordeste: 
há um hub dominante ou o ecossistema está disperso?

> Os municípios são identificados por código da Receita Federal. 
O mapeamento cobre as principais capitais e cidades; demais são exibidos como `Mun. XXXX`.


In [ ]:
# Mapeamento de códigos de município (Receita Federal) → nome
municipio_map = {
    # Sudeste
    '7107': 'São Paulo',        '7047': 'Campinas',         '6897': 'Guarulhos',
    '6001': 'Rio de Janeiro',   '5865': 'Niterói',          '5909': 'São Gonçalo',
    '4123': 'Belo Horizonte',   '5403': 'Uberlândia',       '4733': 'Contagem',
    '5705': 'Vitória',          '5703': 'Cariacica',        '5699': 'Serra',
    # Sul
    '7535': 'Curitiba',         '7691': 'Maringá',          '7667': 'Londrina',
    '8801': 'Porto Alegre',     '8599': 'Caxias do Sul',    '8589': 'Canoas',
    '8105': 'Florianópolis',    '8179': 'Joinville',        '8047': 'Blumenau',
    # Centro-Oeste
    '9701': 'Brasília',         '9373': 'Goiânia',          '9227': 'Anápolis',
    '9051': 'Campo Grande',     '9067': 'Cuiabá',
    # Nordeste
    '3849': 'Salvador',         '3685': 'Feira de Santana', '3515': 'Vitória da Conquista',
    '1389': 'Fortaleza',        '1247': 'Caucaia',          '1447': 'Maracanãu',
    '2531': 'Recife',           '2491': 'Caruaru',          '2457': 'Olinda',
    '2051': 'João Pessoa',      '1981': 'Campina Grande',
    '1761': 'Natal',            '1759': 'Mossoró',
    '3105': 'Aracaju',
    '2785': 'Maceió',
    '1219': 'Teresina',
    '0921': 'São Luís',         '0803': 'Imperatriz',
    # Norte
    '0255': 'Manaus',
    '0427': 'Belém',            '0415': 'Ananindeua',
    '9733': 'Palmas',
    '0003': 'Porto Velho',
    '0605': 'Macapá',
    '0139': 'Rio Branco',
    '0301': 'Boa Vista',
}

def nome_mun(cod):
    return municipio_map.get(str(cod), f'Mun. {cod}')

regioes_ordem = ['Sudeste', 'Sul', 'Nordeste', 'Centro-Oeste', 'Norte']

fig, axes = plt.subplots(len(regioes_ordem), 1, figsize=(12, 4.5 * len(regioes_ordem)))

for ax, regiao in zip(axes, regioes_ordem):
    df_reg = df[df['regiao'] == regiao]
    total_regiao = len(df_reg)
    top = (
        df_reg.groupby(['municipio', 'uf'])
        .size()
        .reset_index(name='n')
        .nlargest(10, 'n')
        .sort_values('n', ascending=True)
    )
    top['pct'] = (top['n'] / total_regiao * 100).round(1)
    top['label'] = top.apply(lambda r: f"{nome_mun(r['municipio'])} ({r['uf']})", axis=1)
    cor = '#f28e2b' if regiao == 'Nordeste' else '#4e79a7'
    bars = ax.barh(top['label'], top['pct'], color=cor)
    ax.bar_label(bars, fmt='%.1f%%', padding=4)
    ax.set_title(f'Top 10 municípios — {regiao}', fontsize=12)
    ax.set_xlabel('% das empresas da região')
    ax.set_xlim(0, top['pct'].max() + 8)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

plt.suptitle('Concentração Municipal de Empresas de Software por Região\n'
             '(% de empresas da região em cada município)', fontsize=14, y=1.005)
plt.tight_layout()
plt.show()


## 1.2 — Concentração na maior cidade por estado

## Visão Geral

Para cada estado, calcula qual porcentagem das empresas está concentrada na maior cidade. 
Dependência de um único polo pode indicar fragilidade do ecossistema — ou que as oportunidades 
ainda não se espalharam pelo interior.


In [ ]:
# Maior cidade por estado
top_por_estado = (
    df.groupby(['uf', 'municipio'])
    .size()
    .reset_index(name='n')
    .sort_values('n', ascending=False)
    .groupby('uf').first().reset_index()
    .rename(columns={'municipio': 'maior_mun', 'n': 'n_maior'})
)

total_por_estado = df.groupby('uf').size().reset_index(name='n_total')
conc = top_por_estado.merge(total_por_estado, on='uf')
conc['pct_capital'] = (conc['n_maior'] / conc['n_total'] * 100).round(1)
conc['nome_cidade'] = conc['maior_mun'].apply(nome_mun)
conc['label'] = conc.apply(lambda r: f"{r['nome_cidade']} / {r['uf']}", axis=1)

nordeste_ufs_set = {'MA','PI','CE','RN','PB','PE','AL','SE','BA'}
conc['is_nordeste'] = conc['uf'].isin(nordeste_ufs_set)
conc = conc.sort_values('pct_capital', ascending=True)

fig, ax = plt.subplots(figsize=(10, 11))
cores = ['#f28e2b' if ne else '#4e79a7' for ne in conc['is_nordeste']]
bars = ax.barh(conc['label'], conc['pct_capital'], color=cores)
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_xlabel('% das empresas do estado na maior cidade')
ax.set_title('Concentração de empresas de software na maior cidade por estado\n'
             '(laranja = estados nordestinos)')
ax.set_xlim(0, conc['pct_capital'].max() + 20)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#f28e2b', label='Nordeste'),
    Patch(color='#4e79a7', label='Demais regiões')
], loc='lower right')
plt.tight_layout()
plt.show()

print('\nTop 5 estados com maior concentração na maior cidade:')
print(conc[['uf','nome_cidade','pct_capital','n_maior','n_total']]
      .sort_values('pct_capital', ascending=False)
      .head(5).to_string(index=False))


# Bloco 2 — Dinâmica de Mortalidade

## 2.1 — Taxa de sobrevivência por coorte de abertura

## Visão Geral

Agrupa empresas por **ano de abertura** e mede quantas ainda estão ativas em fev/2026. 
Permite comparar se o Nordeste tem maior mortalidade — ou se o problema é estrutural (abre menos).

> ⚠ **Nota metodológica:** 2025 foi excluído por ser dado parcial 
(dataset reflete fev/2026; empresas abertas em 2025 ainda não têm situação consolidada).


In [ ]:
coortes = [2008, 2010, 2012, 2015, 2018, 2020, 2022]
df_coorte = df[df['ano_abertura'].isin(coortes)].copy()

sobrev = (
    df_coorte.groupby(['ano_abertura', 'regiao', 'situacao_cadastral'])
    .size().unstack(fill_value=0).reset_index()
)
if '02' not in sobrev.columns:
    sobrev['02'] = 0

cols_status = [c for c in sobrev.columns if c not in ('ano_abertura', 'regiao')]
sobrev['total'] = sobrev[cols_status].sum(axis=1)
sobrev['pct_ativa'] = (sobrev['02'] / sobrev['total'] * 100).round(1)

pivot = sobrev.pivot(index='ano_abertura', columns='regiao', values='pct_ativa')

estilos = {
    'Nordeste':     {'lw': 3.0, 'ls': '-',  'color': '#f28e2b', 'marker': 'o', 'zorder': 5},
    'Sudeste':      {'lw': 1.5, 'ls': '--', 'color': '#4e79a7', 'marker': 's'},
    'Sul':          {'lw': 1.5, 'ls': '--', 'color': '#76b7b2', 'marker': '^'},
    'Centro-Oeste': {'lw': 1.5, 'ls': ':',  'color': '#59a14f', 'marker': 'D'},
    'Norte':        {'lw': 1.5, 'ls': ':',  'color': '#e15759', 'marker': 'v'},
}

fig, ax = plt.subplots(figsize=(12, 5))
for regiao, kwargs in estilos.items():
    if regiao in pivot.columns:
        ax.plot(pivot.index, pivot[regiao], label=regiao, **kwargs)

ax.set_title('Taxa de Sobrevivência por Coorte de Abertura e Região\n'
             '(% de empresas ainda ativas em fev/2026)')
ax.set_xlabel('Ano de abertura')
ax.set_ylabel('% de empresas ativas')
ax.set_xticks(coortes)
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

print('\nTaxa de sobrevivência por coorte e região (%)')
print(pivot.round(1).to_string())


## 2.2 — Perfil das empresas baixadas (fechadas)

## Visão Geral

Analisa o **tempo de vida** das empresas que encerraram atividades (`situacao_cadastral = 08`) 
e quais CNAEs têm maior mortalidade relativa.


In [ ]:
df_baixadas = df[df['situacao_cadastral'] == '08'].copy()

df_baixadas['dt_inicio'] = pd.to_datetime(df_baixadas['data_inicio_atividade'], errors='coerce')
df_baixadas['dt_fim']    = pd.to_datetime(df_baixadas['data_situacao_cadastral'], errors='coerce')
df_baixadas['anos_vida'] = (df_baixadas['dt_fim'] - df_baixadas['dt_inicio']).dt.days / 365.25
df_baixadas = df_baixadas[df_baixadas['anos_vida'].between(0, 50)]

vida_por_regiao = df_baixadas.groupby('regiao')['anos_vida'].median().sort_values()

cnae_baixadas = df_baixadas['cnae_nome'].value_counts()
cnae_total    = df['cnae_nome'].value_counts()
taxa_morte    = (cnae_baixadas / cnae_total * 100).dropna().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cores = ['#f28e2b' if r == 'Nordeste' else '#4e79a7' for r in vida_por_regiao.index]
bars = axes[0].barh(vida_por_regiao.index, vida_por_regiao.values, color=cores)
axes[0].bar_label(bars, fmt='%.1f anos', padding=3)
axes[0].set_title('Tempo Mediano de Vida das Empresas Baixadas por Região')
axes[0].set_xlabel('Anos')

taxa_morte.plot(kind='barh', ax=axes[1], color='#e15759')
axes[1].bar_label(axes[1].containers[0], fmt='%.1f%%', padding=3)
axes[1].set_title('Taxa de Mortalidade por CNAE (% do total no segmento)')
axes[1].set_xlabel('% de empresas baixadas')

plt.tight_layout()
plt.show()

print(f'Total de empresas baixadas: {len(df_baixadas):,} ({len(df_baixadas)/len(df)*100:.1f}% do total)')
print(f'Tempo mediano de vida (geral): {df_baixadas["anos_vida"].median():.1f} anos')
print('\nTempo mediano por região:')
print(vida_por_regiao.round(1).to_string())


# Bloco 3 — Perfil Setorial: Nordeste vs. Sudeste

## 3.1 — Distribuição de CNAE lado a lado

## Visão Geral

Compara o **mix de atividades** entre Nordeste e Sudeste. Hipótese central: o Sudeste concentra mais 
software de **produto** (customizável, não-customizável) enquanto o Nordeste concentra mais 
**serviço** (consultoria, suporte). Essa diferença de modelo explica parte do gap de escala.


In [ ]:
# Consolidar 6201500 e 6201501 (mesmo segmento, códigos distintos)
def norm_cnae(c):
    return '6201500' if c == '6201501' else c

cnae_label_curto = {
    '6201500': 'Desenv. sob encomenda',
    '6201502': 'Web design',
    '6202300': 'Soft. customizável',
    '6203100': 'Soft. não-customizável',
    '6204000': 'Consultoria em TI',
    '6209100': 'Suporte técnico/TI',
}

dados_comp = {}
for reg in ['Nordeste', 'Sudeste']:
    df_r = df[df['regiao'] == reg].copy()
    df_r['cnae_norm'] = df_r['cnae_fiscal_principal'].apply(norm_cnae)
    dados_comp[reg] = df_r['cnae_norm'].value_counts(normalize=True) * 100

df_comp = pd.DataFrame(dados_comp).fillna(0)
df_comp.index = df_comp.index.map(lambda x: cnae_label_curto.get(x, x))
df_comp = df_comp.sort_values('Nordeste', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
y = range(len(df_comp))
h = 0.35

bars_ne = ax.barh([i - h/2 for i in y], df_comp['Nordeste'], h, label='Nordeste', color='#f28e2b')
bars_se = ax.barh([i + h/2 for i in y], df_comp['Sudeste'],  h, label='Sudeste',  color='#4e79a7')

ax.set_yticks(list(y))
ax.set_yticklabels(df_comp.index)
ax.set_xlabel('% das empresas da região')
ax.set_title('Distribuição de CNAE: Nordeste vs. Sudeste')
ax.bar_label(bars_ne, fmt='%.1f%%', padding=2, fontsize=9)
ax.bar_label(bars_se, fmt='%.1f%%', padding=2, fontsize=9)
ax.legend()
ax.set_xlim(0, df_comp.max().max() + 18)
plt.tight_layout()
plt.show()

print('\nDistribuição percentual por CNAE:')
print(df_comp.round(1).to_string())


## 3.2 — Proxies de formalização por região

## Visão Geral

Empresas com **e-mail e telefone cadastrados** na Receita Federal tendem a ser mais formais e operacionalmente ativas. 
Mede a taxa de preenchimento dessas informações por região — proxy complementar de maturidade operacional.


In [ ]:
df['tem_email']    = df['correio_eletronico'].notna() & (df['correio_eletronico'].str.strip() != '')
df['tem_telefone'] = df['telefone1'].notna() & (df['telefone1'].str.strip() != '')

formaliz = (
    df.groupby('regiao')
    .agg(pct_email=('tem_email', 'mean'), pct_tel=('tem_telefone', 'mean'))
    * 100
).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, titulo in zip(
    axes,
    ['pct_email', 'pct_tel'],
    ['E-mail cadastrado', 'Telefone cadastrado']
):
    dados = formaliz[col].sort_values(ascending=True)
    cores = ['#f28e2b' if r == 'Nordeste' else '#4e79a7' for r in dados.index]
    bars = ax.barh(dados.index, dados.values, color=cores)
    ax.bar_label(bars, fmt='%.1f%%', padding=3)
    ax.set_title(f'% de empresas com {titulo}')
    ax.set_xlabel('% das empresas')
    ax.set_xlim(0, 100)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

plt.suptitle('Proxies de Formalização por Região', fontsize=13)
plt.tight_layout()
plt.show()

print('\nTaxas de formalização por região (%)')
print(formaliz.rename(columns={'pct_email': 'E-mail (%)', 'pct_tel': 'Telefone (%)'}).to_string())


# Bloco 4 — Análise Intra-Nordeste

## 4.1 — Ranking dos estados nordestinos

## Visão Geral

O Nordeste não é homogêneo. CE, PE e BA concentram a maior parte das empresas, mas o comportamento varia bastante 
em taxa de atividade, adesão ao Simples e presença de MEI. Revela quais estados têm ecossistema mais maduro.


In [ ]:
nordeste_ufs = ['MA','PI','CE','RN','PB','PE','AL','SE','BA']
nomes_uf = {
    'MA': 'Maranhão', 'PI': 'Piauí',   'CE': 'Ceará',
    'RN': 'Rio G. Norte', 'PB': 'Paraíba', 'PE': 'Pernambuco',
    'AL': 'Alagoas',  'SE': 'Sergipe',   'BA': 'Bahia'
}

df_ne = df[df['uf'].isin(nordeste_ufs)].copy()
df_ne['uf_nome'] = df_ne['uf'].map(nomes_uf)

metricas = df_ne.groupby('uf_nome').agg(
    n_empresas =('cnpj_basico', 'count'),
    pct_ativa  =('situacao_cadastral', lambda x: (x == '02').mean() * 100),
    pct_simples=('opcao_simples',      lambda x: (x == 'S').mean()  * 100),
).round(1)

# Nota: MEI omitido — CNAEs de software (6201-6209) nao constam na lista de
# atividades permitidas para MEI; o registro e nulo em todo o setor.

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titulos = ['Nº de empresas', 'Taxa de atividade (%)', '% Simples Nacional']
colunas = ['n_empresas', 'pct_ativa', 'pct_simples']

for ax, col, titulo in zip(axes, colunas, titulos):
    dados = metricas[col].sort_values(ascending=True)
    bars = ax.barh(dados.index, dados.values, color='#f28e2b')
    if col == 'n_empresas':
        ax.bar_label(bars, labels=[f"{v:,.0f}".replace(',', '.') for v in dados.values], padding=3, fontsize=8)
    else:
        ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)
    ax.set_title(titulo, fontsize=10)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

plt.suptitle('Ranking dos Estados Nordestinos — Setor de Software', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Tabela resumo — estados nordestinos:')
print(metricas.sort_values('n_empresas', ascending=False).to_string())
print('\nNota: MEI nao incluido — atividades de software (CNAEs 6201-6209) nao sao elegiveis para MEI.')


## 4.2 — Crescimento relativo por estado nordestino (2015–2024)

## Visão Geral

Mostra qual estado nordestino está crescendo mais rápido em abertura de novas empresas de software. 
O índice relativo (base 2015 = 100) elimina o efeito de tamanho e revela a dinâmica de crescimento.

> ⚠ **Nota metodológica:** 2025 excluído por ser dado parcial.


In [ ]:
df_ne_time = df[
    df['uf'].isin(nordeste_ufs) &
    df['ano_abertura'].between(2015, 2024)
].copy()
df_ne_time['uf_nome'] = df_ne_time['uf'].map(nomes_uf)

evolucao_ne = (
    df_ne_time.groupby(['ano_abertura', 'uf_nome'])
    .size().unstack(fill_value=0)
)

# Índice relativo (2015 = 100)
base_2015 = evolucao_ne.loc[2015].replace(0, 1)  # evitar divisão por zero
evolucao_idx = (evolucao_ne.div(base_2015) * 100)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absoluto
for col in evolucao_ne.columns:
    axes[0].plot(evolucao_ne.index, evolucao_ne[col], marker='o', label=col)
axes[0].set_title('Abertura anual por estado nordestino (2015–2024)')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Nº de empresas abertas')
axes[0].legend(fontsize=8)
axes[0].set_xticks(range(2015, 2025))
axes[0].tick_params(axis='x', rotation=45)

# Relativo
for col in evolucao_idx.columns:
    lw = 2.5 if col in ('Ceará', 'Pernambuco', 'Bahia') else 1.2
    axes[1].plot(evolucao_idx.index, evolucao_idx[col], marker='o', label=col, linewidth=lw)
axes[1].axhline(100, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('Crescimento relativo (base 2015 = 100)')
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Índice')
axes[1].legend(fontsize=8)
axes[1].set_xticks(range(2015, 2025))
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Dinâmica de Crescimento do Setor de Software nos Estados Nordestinos', fontsize=13)
plt.tight_layout()
plt.show()

cresc = ((evolucao_ne.loc[2024] - evolucao_ne.loc[2015]) / base_2015 * 100).sort_values(ascending=False)
print('\nCrescimento acumulado 2015 -> 2024 por estado nordestino:')
for estado, pct in cresc.items():
    print(f'  {estado:<20} {pct:+.1f}%')


# Bloco 5 — Composição de Porte por Região

## 5.1 — Simples Nacional vs. Fora do Simples

## Visão Geral

O regime tributário é proxy direto de porte e capacidade de escala:
- **Simples Nacional** → pequena/micro empresa (faturamento até R$ 4,8 MM/ano)
- **Fora do Simples** → empresa de médio/grande porte ou que ultrapassou o limite

> **Nota:** MEI não aparece neste setor. As atividades de software (CNAEs 6201–6209) 
**não constam** na lista de ocupações permitidas para registro como Microempreendedor Individual.

Uma região com mais empresas **no Simples** indica ecossistema de menor escala; 
mais **fora do Simples** indica maior presença de empresas médias/grandes.


In [ ]:
# CNAEs de software nao sao elegiveis para MEI — porte e binario: Simples vs Fora
df['porte'] = df['opcao_simples'].map({'S': 'Simples Nacional', 'N': 'Fora do Simples'})

porte_regiao = df.groupby(['regiao', 'porte']).size().unstack(fill_value=0)
porte_pct    = (porte_regiao.div(porte_regiao.sum(axis=1), axis=0) * 100).round(1)

if 'Simples Nacional' in porte_pct.columns:
    porte_pct = porte_pct.sort_values('Simples Nacional', ascending=True)

cores_porte = {'Simples Nacional': '#f28e2b', 'Fora do Simples': '#4e79a7'}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Percentual
ax1 = axes[0]
bottom = pd.Series(0.0, index=porte_pct.index)
for porte in ['Fora do Simples', 'Simples Nacional']:
    if porte in porte_pct.columns:
        ax1.barh(porte_pct.index, porte_pct[porte], left=bottom,
                 label=porte, color=cores_porte[porte])
        for i, (val, bot) in enumerate(zip(porte_pct[porte], bottom)):
            if val > 3:
                ax1.text(bot + val / 2, i, f'{val:.1f}%',
                         ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        bottom += porte_pct[porte]
ax1.set_xlabel('% das empresas')
ax1.set_title('Composição de porte por região (%)')
ax1.set_xlim(0, 105)
ax1.legend(loc='lower right')

# Absoluto
ax2 = axes[1]
porte_abs  = porte_regiao.reindex(porte_pct.index)
bottom_abs = pd.Series(0.0, index=porte_abs.index)
for porte in ['Fora do Simples', 'Simples Nacional']:
    if porte in porte_abs.columns:
        ax2.barh(porte_abs.index, porte_abs[porte], left=bottom_abs,
                 label=porte, color=cores_porte[porte])
        bottom_abs += porte_abs[porte]
ax2.set_xlabel('Nº de empresas')
ax2.set_title('Composição de porte por região (absoluto)')
ax2.legend(loc='lower right')

plt.suptitle(
    'Perfil de Porte por Regime Tributário — Setor de Software\n'
    '(MEI não aplicável: CNAEs de software não constam na lista de atividades permitidas para MEI)',
    fontsize=12
)
plt.tight_layout()
plt.show()

print('Composição percentual por região:')
print(porte_pct.to_string())
print('\nNúmeros absolutos:')
print(porte_regiao.reindex(porte_pct.index).to_string())
